# Structured Outputs with OpenAI API

In this notebook, you'll learn how to get **structured, type-safe JSON responses** from OpenAI's API using the Responses API.


You'll be able to:

1. **Understand response formats** - The modern way to get structured JSON from LLMs
2. **Use JSON Schema** - Define exact structure and types for your outputs
3. **Handle complex data** - Work with nested objects, arrays, and nullable fields
4. **Apply best practices** - Avoid common pitfalls and build production-ready systems
5. **Build real applications** - Create practical tools like event extractors and data parsers





---

# Setup

## Install Dependencies

In [ ]:
# Install required packages
!pip install -q openai==2.28.0

print("✅ All dependencies installed!")

## API Key Configuration

You have two methods to provide your API key:

**Method 1 (Recommended)**: Use Colab Secrets
1. Click the 🔑 icon in the left sidebar
2. Click "Add new secret"
3. Name: `OPENAI_API_KEY`
4. Value: Your OpenAI API key
5. Enable notebook access

**Method 2 (Fallback)**: Manual input when prompted

Run the cell below to configure authentication:

In [ ]:
import os

# Configure OpenAI API key
# Method 1: Try to get API key from Colab secrets (recommended)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    # Method 2: Manual input (fallback)
    from getpass import getpass
    print("💡 To use Colab secrets: Go to 🔑 (left sidebar) → Add new secret → Name: OPENAI_API_KEY")
    OPENAI_API_KEY = getpass("Enter your OpenAI API Key: ")

# Set the API key as an environment variable
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if not OPENAI_API_KEY or OPENAI_API_KEY.strip() == "":
    raise ValueError("❌ ERROR: No API key provided!")

print("✅ Authentication configured!")

OPENAI_MODEL = "gpt-5-nano"
print(f"Selected Model: {OPENAI_MODEL}")

## Import Libraries

In [ ]:
import json
from openai import OpenAI

print("✅ All libraries imported!")

## Initialize OpenAI Client

Now let's create a client instance to interact with the OpenAI API:

In [ ]:
# Initialize OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ OpenAI client initialized!")

---

# Introduction to Structured Outputs

## Two Ways to Get Structured Data

OpenAI offers two main approaches to get structured outputs from language models:

### 1. **Function Calling**
- The model requests function execution
- You parse the function arguments
- Useful when you want the model to trigger actions

### 2. **Response Formats** (Focus of This Notebook)
- The model responds **directly in structured JSON**
- No function calls - just get the data you need
- Simpler for pure data extraction tasks

Response formats are perfect when you want to extract information from text, parse unstructured data into structured forms, or generate structured content like product listings and event details.

The key benefits of structured outputs are type safety (guaranteed data types), consistency (same field names and structure every time), explicit refusals (the model can decline when appropriate), simpler prompting (no need for detailed formatting instructions), and production readiness (reliable enough for real applications).



---

# Response Formats - Getting Structured JSON

Let's explore how to get structured JSON responses from the OpenAI Responses API. We'll start with the old way, then move to the modern, better approach!

## The Old Way - Prompting for JSON

Before structured outputs became available, the only way to get JSON responses from language models was to simply ask for it in your prompt. You would write something like "return the result as JSON" and hope the model would comply. While this approach often worked, it came with significant limitations that made it unsuitable for production applications.

The fundamental issue is that **you have no control over the output structure**. The model interprets your instructions probabilistically, and each call might produce slightly different results.

- **Inconsistent field names** - Even when you explicitly specify field names, the model might use variations. A "price" field might come back as `"price"`, `"cost"`, or `"product_price"` in different responses.

- **Unpredictable data types** - A price value might come back as a number (`49.99`) in one response and as a string (`"$49.99"`) in another. The same applies to booleans (might get `true`, `"true"`, `"yes"`, or `1`).

- **Missing or extra fields** - The model might omit fields you requested or add extra fields you did not ask for. You cannot rely on a consistent structure when processing the responses.

- **Structural variations** - Sometimes the model returns a flat object, other times it nests related fields into sub-objects. For example, you might get `{"price": 49.99, "currency": "USD"}` in one response and `{"price": {"amount": 49.99, "currency": "USD"}}` in another.

- **Invalid JSON** - While modern models are generally good at producing valid JSON syntax, there is still no guarantee, especially with complex nested structures.

These inconsistencies are not bugs - they are inherent to how language models work. Models generate text probabilistically, token by token. Without explicit constraints, the model makes independent decisions about field names, types, and structure based on its training and the context.

For production applications, this means unreliable parsing, fragile integrations with APIs and databases, difficult debugging, and no type safety. Developers end up writing extensive post-processing code to check multiple field name variations, convert types, and handle unexpected structures.

## The Modern Way - `json_schema` with Responses API



Because of these limitations with prompt-based JSON, OpenAI introduced **structured outputs** using JSON Schema. This feature fundamentally changes the game by allowing you to define an explicit schema that the model must follow. Instead of hoping the model interprets your prompt correctly, you provide a formal specification that guarantees consistency.

With structured outputs, you define exactly what fields you want, their data types, which fields are required, and even constraints like enums for fixed choices. The model then generates outputs that strictly conform to your schema—every single time. This transforms language models from unpredictable text generators into reliable structured data extractors that you can build production systems on.

### The Responses API Syntax

The Responses API uses a specific syntax for structured outputs. Instead of the `response_format` parameter used in Chat Completions, you use the `text` parameter with a nested `format` object:

```python
text={
    "format": {
        "type": "json_schema",
        "name": "schema_name",
        "schema": your_schema,
        "strict": True
    }
}
```

This syntax is unique to the Responses API and is essential for getting structured outputs with models like `gpt-5-nano`.

### The Magic Parameter: `strict: True`

The key to reliable structured outputs is the `strict: True` parameter. This is what transforms the API from "try to follow this schema" to "must follow this schema exactly." When you set `strict: True`:

- **Field names are guaranteed** - Always exactly as you specified, never variations
- **Data types are enforced** - Numbers are always numbers, strings are always strings, booleans are always booleans
- **Required fields are present** - Every field you mark as required will be in the response
- **No unexpected fields** - When using `additionalProperties: false`, only your defined fields appear
- **Structure is consistent** - The same schema produces the same structure every time

Without `strict: True`, you're back to the probabilistic behavior of the old way. With it, you get deterministic, type-safe outputs.

### ✅ Key Benefits Over Prompt-Based JSON

Let's compare what structured outputs give you:

| Aspect | Prompt-Based JSON ❌ | Structured Outputs ✅ |
|--------|---------------------|---------------------|
| Field names | Vary between calls | Always consistent |
| Data types | Unpredictable | Guaranteed |
| Required fields | May be missing | Always present |
| Extra fields | May appear | Prevented with additionalProperties: false |
| Structure | Can vary | Fixed by schema |
| Type safety | Impossible | Full support |
| Production ready | No | Yes |

### How It Works

When you use structured outputs with JSON Schema:

1. **Define your schema** - Specify the exact structure you want using JSON Schema format
2. **Set strict mode** - Include `strict: True` to enforce the schema
3. **Make the API call** - The model generates content that conforms to your schema
4. **Parse with confidence** - You can trust the structure and types of the response

The model understands your schema and generates tokens that are guaranteed to produce valid output matching your specification. This isn't post-processing or validation—it's built into the generation process itself.

### Ready for Production

With structured outputs, you can:
- Write parsing code once and trust it will always work
- Integrate directly with databases and APIs
- Use static type checking in your application
- Validate data using your schema
- Build reliable, maintainable systems

Let's see this in action with a practical example!

In [ ]:
# Example: Extract product information using json_schema
# ✅ This is the MODERN way - notice the consistency!

product_description = """
The TechPro Wireless Mouse is a premium ergonomic mouse perfect for office work.
It's priced at $49.99 and falls under the Electronics category.
Features include 2400 DPI precision and 18-month battery life.
"""

print("🔍 Extracting product information using json_schema...")

# Define our schema - exactly what we want!
product_schema = {
    "type": "object",  # The response is a JSON object
    "properties": {  # Define each field
        "name": {
            "type": "string",  # Product name as text
            "description": "The name of the product"
        },
        "price": {
            "type": "number",  # Price as a number (not string!)
            "description": "The price in USD as a number"
        },
        "category": {
            "type": "string",  # Category as text
            "description": "The product category"
        },
        "description": {
            "type": "string",  # Description as text
            "description": "Brief product description"
        }
    },
    "required": ["name", "price", "category", "description"],  # All fields required
    "additionalProperties": False  # No extra fields allowed
}

# Make API call with json_schema format using Responses API syntax
# IMPORTANT: Responses API uses text.format, not response_format!
response = client.responses.create(
    model=OPENAI_MODEL,
    input=f"Extract product information: {product_description}",
    text={  # Use text parameter (not response_format)
        "format": {  # Nested format object
            "type": "json_schema",  # Type is json_schema
            "name": "product_extraction",  # Schema name
            "schema": product_schema,  # Our defined schema
            "strict": True  #  THE MAGIC PARAMETER - enforce schema strictly!
        }
    }
)

# Parse the JSON response
product_data = json.loads(response.output_text)

print("Extracted Product Data:")
print(json.dumps(product_data, indent=2))

print("✅ Run this cell multiple times - the output is ALWAYS consistent!")
print("   ✓ Same field names every time")
print("   ✓ Price is always a number, not a string")
print("   ✓ All required fields are present")
print("   ✓ No unexpected extra fields")

### Notice the Difference!

✅ **Consistent structure**:
```json
{
  "name": "TechPro Wireless Mouse",
  "price": 49.99,
  "category": "Electronics",
  "description": "Premium ergonomic mouse perfect for office work"
}
```

Run it 100 times - you'll get the **exact same structure** every single time!

### Responses API Syntax Recap:

```python
response = client.responses.create(
    model="gpt-5-nano",
    input="your prompt",
    text={                          # Use 'text' parameter
        "format": {                 # Nested 'format' object
            "type": "json_schema",  # Type is json_schema
            "name": "schema_name",  # Give your schema a name
            "schema": {...},        # Your JSON schema
            "strict": True          # Always use strict mode!
        }
    }
)
```




## Exercise: Extract Book Information

Now it's your turn! Modify the code above to extract book information with these fields:
- **title** (string) - The book title
- **author** (string) - The author's name
- **year_published** (integer) - Year as a number
- **genres** (array of strings) - List of genre tags

### Test Input:

```
The book "1984" was written by George Orwell and published in 1949.
It's a dystopian fiction novel that also falls under political fiction and science fiction genres.
```

### Expected Output Structure:

```json
{
  "title": "1984",
  "author": "George Orwell",
  "year_published": 1949,
  "genres": ["dystopian fiction", "political fiction", "science fiction"]
}
```

### 💡 Hints:
- Use `"type": "integer"` for the year
- Use `"type": "array"` with `"items": {"type": "string"}` for the genres
- Don't forget `"strict": True` and `"additionalProperties": False`
- Remember to use `text={"format": {...}}` syntax!

In [ ]:
# YOUR CODE HERE
# Define the book schema and extract information from the test input

book_description = """
The book "1984" was written by George Orwell and published in 1949.
It's a dystopian fiction novel that also falls under political fiction and science fiction genres.
"""

# Define your book_schema here
book_schema = {
    # TODO: Fill in the schema
}

# Make the API call
# TODO: Complete the API call with json_schema format using text parameter

## Understanding JSON Schema Basics

JSON Schema supports several fundamental data types. Choosing the right type for each field is important for getting reliable structured outputs.

### Core Data Types

#### 1. **`string`** - Text data
- Names, descriptions, URLs, emails, etc.
- Example: `"John Doe"`, `"hello@example.com"`

#### 2. **`integer`** - Whole numbers
- Years, counts, IDs, quantities
- Example: `2024`, `42`, `-5`

#### 3. **`number`** - Decimal numbers
- Prices, ratings, measurements, percentages
- Example: `49.99`, `3.14`, `-2.5`

#### 4. **`boolean`** - True/false values
- Flags, status indicators, yes/no questions
- Example: `true`, `false`

#### 5. **`array`** - Lists of items
- Collections of things (tags, names, values)
- Must specify what type of items: `"items": {"type": "string"}`
- Example: `["red", "green", "blue"]`

#### 6. **`object`** - Nested structures
- Complex data with its own fields
- Example: `{"street": "123 Main St", "city": "Boston"}`

---

### Required Fields

The `required` array specifies which fields must be present in every response:

```json
"required": ["name", "email", "age"]
```

---

### Additional Properties

The `additionalProperties` setting controls whether the model can add extra fields beyond what you defined:

```json
"additionalProperties": false
```

Always set this to `false` for predictable outputs.

---

Let's see all these types in action!

In [ ]:
# Example: Extract movie information demonstrating all data types

movie_description = """
The movie "Inception" was released in 2010 and belongs to the Science Fiction genre.
It has an IMDb rating of 8.8 out of 10 and runs for 148 minutes.
The film is available on Netflix streaming.
"""

print("Extracting movie information with various data types...")

# Define schema demonstrating different data types
movie_schema = {
    "type": "object",
    "properties": {
        "title": {
            "type": "string",  # Text - movie title
            "description": "The movie title"
        },
        "year": {
            "type": "integer",  # Whole number - release year
            "description": "Release year as an integer"
        },
        "genre": {
            "type": "string",  # Text - genre category
            "description": "Primary genre"
        },
        "rating": {
            "type": "number",  # Decimal number - rating score
            "description": "IMDb rating as a decimal number"
        },
        "runtime_minutes": {
            "type": "integer",  # Whole number - duration in minutes
            "description": "Runtime in minutes as an integer"
        },
        "streaming_available": {
            "type": "boolean",  # True/false - availability flag
            "description": "Whether the movie is available on streaming platforms"
        }
    },
    "required": ["title", "year", "genre", "rating", "runtime_minutes", "streaming_available"],
    "additionalProperties": False  # No extra fields allowed
}

# Make API call using Responses API syntax
response = client.responses.create(
    model=OPENAI_MODEL,
    input=f"Extract movie information: {movie_description}",
    text={
        "format": {
            "type": "json_schema",
            "name": "movie_extraction",
            "schema": movie_schema,
            "strict": True  # Enforce schema strictly
        }
    }
)

# Parse and display
movie_data = json.loads(response.output_text)

print("Extracted Movie Data:")
print(json.dumps(movie_data, indent=2))

# Show the types
print(" Data Types Verification:")
print(f"   title type: {type(movie_data['title']).__name__} (string) ✓")
print(f"   year type: {type(movie_data['year']).__name__} (integer) ✓")
print(f"   genre type: {type(movie_data['genre']).__name__} (string) ✓")
print(f"   rating type: {type(movie_data['rating']).__name__} (number/float) ✓")
print(f"   runtime_minutes type: {type(movie_data['runtime_minutes']).__name__} (integer) ✓")
print(f"   streaming_available type: {type(movie_data['streaming_available']).__name__} (boolean) ✓")

### Type Selection Guide

| Data | Use This Type | Why |
|------|---------------|-----|
| Names, descriptions, URLs | `string` | Text information |
| Years, counts, IDs | `integer` | Whole numbers only |
| Prices, ratings, percentages | `number` | May have decimals |
| Yes/no, true/false, flags | `boolean` | Binary states |
| Lists of items | `array` | Multiple values |
| Complex nested data | `object` | Structured information |



---

# Practical Example - Event Information Extractor

## Scenario: Building a Calendar Assistant

Imagine you're building a smart calendar assistant that reads event descriptions from emails, messages, or meeting notes and automatically creates calendar entries. Users can write naturally about their events, and your system extracts all the structured details needed!

### The Goal:

Extract structured event information from natural language descriptions:
- Event title and description
- Date and time details
- Location (physical or virtual)
- Attendee list
- Meeting links for virtual events


Real-world data is messy! People describe events in different ways:
- "Let's meet tomorrow at 3pm for an hour"
- "Conference call on Zoom next Tuesday, link in email"
- "Team lunch at the office cafeteria, bring everyone"

Your system needs to handle all these variations and extract consistent, structured data.

Let's build it!

In [ ]:
# Practical Example: Event Information Extractor

# Sample event description (natural language)
event_text = """
We have a quarterly planning meeting scheduled for March 15, 2024, starting at 2:00 PM.
The session will last approximately 90 minutes and will be held virtually via Zoom.
The meeting link is https://zoom.us/j/123456789.
Key attendees include Sarah Chen, Mike Rodriguez, and Jennifer Liu.
We'll be discussing Q2 goals, budget allocation, and team restructuring plans.
"""

print("Extracting event information...")

# Define comprehensive event schema
event_schema = {
    "type": "object",
    "properties": {
        # Basic event information
        "event_title": {
            "type": "string",
            "description": "The title or name of the event"
        },
        "description": {
            "type": "string",
            "description": "Brief description of what will be discussed or done"
        },

        # Date and time information
        "date": {
            "type": "string",
            "description": "Date in YYYY-MM-DD format"
        },
        "start_time": {
            "type": "string",
            "description": "Start time in HH:MM format (24-hour)"
        },
        "duration_minutes": {
            "type": "integer",  # Integer for whole number of minutes
            "description": "Duration in minutes"
        },

        # Location information
        "location": {
            "type": "string",
            "description": "Physical location or 'Virtual' if online"
        },
        "is_virtual": {
            "type": "boolean",  # Boolean for true/false flag
            "description": "Whether the meeting is virtual/online"
        },

        # Nullable field example: meeting link might not always exist
        "meeting_link": {
            "type": ["string", "null"],  # Can be string OR null
            "description": "Virtual meeting link if applicable, null otherwise"
        },

        # Array of attendees
        "attendees": {
            "type": "array",  # Array type for lists
            "items": {  # Define what each item in the array should be
                "type": "string"
            },
            "description": "List of attendee names"
        }
    },
    # All fields are required
    "required": [
        "event_title",
        "description",
        "date",
        "start_time",
        "duration_minutes",
        "location",
        "is_virtual",
        "meeting_link",
        "attendees"
    ],
    "additionalProperties": False  # No extra fields
}

# Extract event information using Responses API
response = client.responses.create(
    model=OPENAI_MODEL,
    input=f"Extract structured event information from this description: {event_text}",
    text={
        "format": {
            "type": "json_schema",
            "name": "event_extraction",
            "schema": event_schema,
            "strict": True
        }
    }
)

# Parse the result
event_data = json.loads(response.output_text)

# Display the extracted information
print("Extracted Event Details:")
print("=" * 60)
print(json.dumps(event_data, indent=2))
print("=" * 60)

# Demonstrate how you'd use this in a real application
print("✅ Ready for Calendar Integration:")
print(f"✓ Event: {event_data['event_title']}")
print(f"✓ When: {event_data['date']} at {event_data['start_time']} ({event_data['duration_minutes']} min)")
print(f"✓ Where: {event_data['location']} {'(Virtual)' if event_data['is_virtual'] else '(In-person)'}")
if event_data['meeting_link']:
    print(f"✓ Link: {event_data['meeting_link']}")
print(f"✓ Attendees: {', '.join(event_data['attendees'])}")
print(f"💡 This structured data can now be sent directly to any calendar API!")

In [ ]:
# YOUR CODE HERE
# Enhance the event schema with the three new fields

enhanced_event_text = """
High-priority client presentation on March 20, 2024 at 10:00 AM, lasting 2 hours.
This is a critical meeting at the downtown office, Conference Room A.
Team members should review the proposal deck beforehand.
Set a reminder 15 minutes before.
Attendees: Alex Kim, Jordan Park, and Taylor Morgan.
"""

# TODO: Copy the event_schema from above and add the three new fields
enhanced_event_schema = {
    # Add your enhanced schema here
}

# TODO: Make the API call and display results using text parameter

---

# Working with Nested Structures



Real-world data is often hierarchical - objects containing other objects or arrays of objects. Nested structures let you represent complex, related information accurately.

### When You Need Nested Structures:

#### 1. **E-commerce Product with Variants**
```
Product: T-Shirt
├─ Variants:
│  ├─ Small (Blue, $19.99, 45 in stock)
│  ├─ Medium (Red, $19.99, 30 in stock)
│  └─ Large (Black, $21.99, 50 in stock)
```
Each variant has its own color, price, and inventory - that's a nested structure!

#### 2. **Organizational Chart**
```
Company: TechCorp
├─ Employees:
│  ├─ CEO (name, hire_date, salary)
│  │  └─ Reports: [list of employee IDs]
│  ├─ CTO (name, hire_date, salary)
│  │  └─ Reports: [list of employee IDs]
```
Employees have their own properties and relationships - nested objects!

#### 3. **Flight Booking**
```
Booking:
├─ Flight: (airline, number, date)
├─ Passengers:
│  ├─ Passenger 1 (name, age, seat)
│  │  └─ Baggage: [checked: 2 bags, carry-on: 1 bag]
│  ├─ Passenger 2 (name, age, seat)
│  │  └─ Baggage: [checked: 1 bag, carry-on: 1 bag]
```
Each passenger has their own details and baggage - complex nested data!

### 💡 Key Concept:

**Nested structures = Objects within objects, or arrays of objects**
- Use when data has hierarchical relationships
- Use when you have collections of complex items (not just simple strings or numbers)
- Essential for modeling real-world entities accurately

Let's see how to define and work with these structures!

In [ ]:
# Example: Extract company information with nested structures

company_description = """
TechStart Inc. is a software development company founded in 2018 in the technology industry.
The company has 150 employees and generates $25 million in annual revenue.

The leadership team includes:
- Sarah Johnson, CEO, who has been in her role for 5 years
- Michael Chen, CTO, who has been in his role for 4 years
- Emily Rodriguez, CFO, who has been in her role for 3 years

TechStart has offices in three locations:
- San Francisco, USA (headquarters)
- Austin, USA (not headquarters)
- Toronto, Canada (not headquarters)
"""

print("Extracting company information with nested structures...")

# Define schema with nested structures
company_schema = {
    "type": "object",
    "properties": {
        # Simple top-level fields
        "company_name": {
            "type": "string",
            "description": "The company name"
        },
        "founded_year": {
            "type": "integer",
            "description": "Year the company was founded"
        },
        "industry": {
            "type": "string",
            "description": "Primary industry"
        },
        "employee_count": {
            "type": "integer",
            "description": "Number of employees"
        },
        "annual_revenue_millions": {
            "type": "number",
            "description": "Annual revenue in millions of dollars"
        },

        # NESTED STRUCTURE 1: Array of objects (leadership team)
        # This is an array where each item is a complex object with multiple fields
        "leadership": {
            "type": "array",  # It's an array...
            "items": {  # ...where each item is:
                "type": "object",  # An object (not just a string!)
                "properties": {  # With its own fields:
                    "name": {
                        "type": "string",
                        "description": "Leader's full name"
                    },
                    "title": {
                        "type": "string",
                        "description": "Job title (CEO, CTO, etc.)"
                    },
                    "years_in_role": {
                        "type": "integer",
                        "description": "Years in current position"
                    }
                },
                "required": ["name", "title", "years_in_role"],  # Required fields for each leader
                "additionalProperties": False
            },
            "description": "List of company leadership members"
        },

        # NESTED STRUCTURE 2: Array of objects (office locations)
        # Another array of complex objects
        "locations": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name"
                    },
                    "country": {
                        "type": "string",
                        "description": "Country name"
                    },
                    "is_headquarters": {
                        "type": "boolean",
                        "description": "Whether this is the headquarters location"
                    }
                },
                "required": ["city", "country", "is_headquarters"],
                "additionalProperties": False
            },
            "description": "List of office locations"
        }
    },
    "required": [
        "company_name",
        "founded_year",
        "industry",
        "employee_count",
        "annual_revenue_millions",
        "leadership",
        "locations"
    ],
    "additionalProperties": False
}

# Extract company information using Responses API
response = client.responses.create(
    model=OPENAI_MODEL,
    input=f"Extract structured company information: {company_description}",
    text={
        "format": {
            "type": "json_schema",
            "name": "company_extraction",
            "schema": company_schema,
            "strict": True
        }
    }
)

# Parse and display
company_data = json.loads(response.output_text)

print("Extracted Company Data:")
print("=" * 70)
print(json.dumps(company_data, indent=2))
print("=" * 70)

# Demonstrate accessing nested data
print(" Accessing Nested Data:")
print(f" Leadership Team ({len(company_data['leadership'])} members):")
for leader in company_data['leadership']:
    print(f"   • {leader['name']} - {leader['title']} ({leader['years_in_role']} years)")

print(f" Office Locations ({len(company_data['locations'])} locations):")
for location in company_data['locations']:
    hq_marker = "⭐ HQ" if location['is_headquarters'] else "📍"
    print(f"   {hq_marker} {location['city']}, {location['country']}")

print("\n💡 Notice how we can easily loop through nested arrays and access their fields!")

### Understanding Nested Structures:

#### Array of Objects Pattern:

```json
{
  "type": "array",          // It's an array
  "items": {                 // Each item in the array is:
    "type": "object",        // An object
    "properties": {          // With these fields:
      "field1": {"type": "string"},
      "field2": {"type": "integer"}
    },
    "required": ["field1", "field2"],
    "additionalProperties": false
  }
}
```

#### Key Points:

1. **Array of Simple Values** (strings, numbers):
   ```json
   "tags": {
     "type": "array",
     "items": {"type": "string"}  // Just specify the type
   }
   ```
   Result: `["tech", "startup", "AI"]`

2. **Array of Objects** (complex items):
   ```json
   "employees": {
     "type": "array",
     "items": {
       "type": "object",         // Each item is an object
       "properties": {...},       // Define its structure
       "required": [...]
     }
   }
   ```
   Result: `[{"name": "Alice", "role": "Dev"}, {"name": "Bob", "role": "PM"}]`

3. **Required Fields Apply to Each Object**:
   - Set `required` inside the `items` definition
   - Every object in the array must have those fields

4. **Use `additionalProperties: false`** at both levels:
   - Top level (main object)
   - Nested level (objects within arrays)

### Benefits:

With nested structures, you can model complex entities accurately:
- ✅ Each leader has consistent fields (name, title, years)
- ✅ Easy to loop through and access
- ✅ Type-safe - years is always an integer
- ✅ No parsing headaches!


---

# Conclusion

You now know how to get reliable, type-safe structured outputs from OpenAI's Responses API. Here is a summary of the most important points:

**Responses API syntax** - Use `text={"format": {...}}` with `"type": "json_schema"` and always include `"strict": True`. This is what makes structured outputs reliable.

**Choose specific types** - Use `integer` for whole numbers, `number` for decimals, `boolean` for true/false, `array` for lists, and `object` for nested data. The more specific you are, the more reliable your outputs will be.

**Set `additionalProperties: false`** at every level (top-level and inside nested objects) to prevent unexpected fields.

**Use nullable types** (`["string", "null"]`) for fields that might not have a value, rather than making them optional.

**Start simple and iterate** - Begin with basic schemas and add complexity gradually. Test with real data, not just perfect examples. Add error handling with try/except for production code.

---

### Additional Resources

- [OpenAI Structured Outputs Guide](https://platform.openai.com/docs/guides/structured-outputs)
- [JSON Schema Documentation](https://json-schema.org/)
- [OpenAI API Reference](https://platform.openai.com/docs/api-reference)